# Agentic AI - Reflection in a Research Agent



In this notbook, you’ll implement a simple **[agentic](https://class.vision/blog/ai-agents/) workflow** designed to simulate reflective thinking in a writing task. This is one building block of a more complex research agent that will be constructed throughout the course.

### 📘 Objective

Build a three-step workflow where an [LLM](https://class.vision/blog/llm-%D9%85%D8%AF%D9%84-%D8%B2%D8%A8%D8%A7%D9%86%DB%8C/) writes an essay draft, critiques it, and rewrites it. The focus of the lab is not on the content quality of the essay, but rather on how you orchestrate the **calls to the LLM** and pass intermediate results between steps.

### 🛠️ What You’ll Build

* **Step 1 – Drafting:** Call the LLM to generate an initial draft of an essay based on a simple prompt.
* **Step 2 – Reflection:** Reflect on the draft using a reasoning step. (Optionally, this can be done with a different model.)
* **Step 3 – Revision:** Apply the feedback from the reflection to generate a revised version of the essay.


## ⚙️ Loading Essentials

Before interacting with the language models, we initialize the `aisuite` client. This setup loads environment variables (e.g., API keys) from a `.env` file to securely authenticate with backend services. The `ai.Client()` instance will be used to make all model calls throughout this workflow.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

In [2]:
if 'COLAB_GPU' in os.environ or not os.path.exists('utils') or not os.path.exists('data'):
    print("📥 Downloading required files...")
    !wget -q https://raw.githubusercontent.com/Alireza-Akhavan/Agentic_AI/refs/heads/main/utils/utils.py -P utils
    !pip install -q aisuite
    print("✅ Setup completed")
else:
    print("✅ Running locally - using existing files")

✅ Running locally - using existing files


In [3]:
import aisuite as ai

#client = ai.Client()

client = ai.Client(
    {
        "openai": {
            "api_key": openai_api_key,
            "base_url": openai_base_url,
        }
    }
)

### 📝 `generate_draft` Function

**Objective**:
Write a function called `generate_draft` that takes in a string prompt and uses a language model to generate a complete draft essay.

**Inputs**:

* `prompt` (str): The essay question or writing prompt.
* `model` (str, optional): The model identifier to use. Defaults to `"openai:gpt-5.1"`.

**Output**:

* A string representing the full draft of the essay.



In [4]:
def generate_draft(prompt: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
                    Write a well-structured draft essay in response to the following prompt.
                    The draft should include an introduction, body, and conclusion.

                    Prompt:
                    {prompt}
                    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content


### 🔍 `reflect_on_draft` Function

**Objective**:
Write a function called `reflect_on_draft` that takes a previously generated essay draft and uses a language model to provide constructive feedback.

**Inputs**:

* `draft` (str): The essay text to reflect on.
* `model` (str, optional): The model identifier to use. Defaults to `"openai:gpt-5.1"`.

**Output**:

* A string with feedback in paragraph form.


In [5]:
def reflect_on_draft(draft: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
                    Reflect critically on the following draft essay.
                    Identify areas for improvement in structure, clarity, argument strength, or style.
                    Provide constructive feedback in paragraph form.

                    Draft:
                    {draft}
                    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content


### 🔁 `revise_draft` Function

**Objective**:
Implement a function called `revise_draft` that improves a given essay draft based on feedback from a reflection step.

**Inputs**:

* `original_draft` (str): The initial version of the essay.
* `reflection` (str): Constructive feedback or critique on the draft.
* `model` (str, optional): The model identifier to use. Defaults to `"openai:gpt-5.1"`.

**Output**:

* A string containing the revised and improved essay.

In [6]:
def revise_draft(original_draft: str, reflection: str, model: str = "openai:gpt-5.1") -> str:
    instruction = f"""
                    Revise the following essay draft using the feedback provided.
                    Make improvements in clarity, coherence, argumentation, and flow.
                    Return only the improved essay.

                    Original Draft:
                    {original_draft}

                    Feedback:
                    {reflection}
                    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=1.0,
    )
    return response.choices[0].message.content


### 🧪 Test the Reflective Writing Workflow

Use the functions you implemented to simulate the complete writing workflow:

1. **Generate a draft** in response to the essay prompt.
2. **Reflect** on the draft to identify improvements.
3. **Revise** the draft using the feedback.


In [26]:
essay_prompt = "آیا شبکه‌های اجتماعی باید توسط دولت تنظیم‌گری (قانون‌گذاری و نظارت) شوند؟"

# Agente 1 – Draft
draft = generate_draft(essay_prompt)
print("📝 Draft:\n")
print(draft)

# Agente 2 – Reflection
feedback = reflect_on_draft(draft)
print("\n🧠 Feedback:\n")
print(feedback)

# Agente 3 – Revision
revised = revise_draft(draft, feedback)
print("\n✍️ Revised:\n")
print(revised)



📝 Draft:

**مقدمه**  
رشد سریع شبکه‌های اجتماعی در دو دهۀ اخیر، نحوۀ ارتباط، اطلاع‌رسانی و حتی شکل‌گیری افکار عمومی را دگرگون کرده است. این فضا از یک‌سو فرصت‌های بی‌سابقه‌ای برای آزادی بیان، مشارکت مدنی، کسب‌وکار و آموزش فراهم کرده و از سوی دیگر، با چالش‌هایی مانند نشر نفرت، شایعات، نقض حریم خصوصی و سوءاستفادۀ سیاسی همراه بوده است. در چنین زمینه‌ای، این پرسش به‌طور جدی مطرح می‌شود که آیا دولت‌ها باید در حوزۀ شبکه‌های اجتماعی قانون‌گذاری و نظارت کنند یا خیر. در این نوشتار، ابتدا دلایل موافقان تنظیم‌گری دولتی بررسی می‌شود، سپس به استدلال‌های مخالفان پرداخته و در پایان، جمع‌بندی و موضعی میانه‌رو ارائه خواهد شد.

---

**بدنۀ اصلی**

### ۱. استدلال‌های موافقان تنظیم‌گری دولت

1. **حفاظت از امنیت ملی و نظم عمومی**  
شبکه‌های اجتماعی می‌توانند ابزاری برای سازمان‌دهی خشونت، تروریسم، نفرت‌پراکنی قومی و مذهبی و آشوب‌های هدفمند باشند. دولت‌ها معمولاً مسئول حفظ امنیت و ثبات‌اند و بدون ابزارهای قانونی برای نظارت و مداخله محدود، ممکن است در برابر تهدیدهای سازمان‌یافته در فضای مجازی ناتوان شوند. برای

To better visualize the output of each step in the reflective writing workflow, we use a utility function called `show_output`. This function displays the results of each stage (drafting, reflection, and revision) in styled boxes with custom background and text colors, making it easier to compare and understand the progression of the essay.


In [46]:
from utils.utils import show_output


show_output("مرحله اول: پیش‌نویس", draft, background="#fff8dc", text_color="#333333")
show_output("مرحله دوم – Reflection", feedback, background="#e0f7fa", text_color="#222222")
show_output("مرحله سوم – بازنگری یا Revision", revised, background="#f3e5f5", text_color="#222222")
